# 02 OPERA DISP-S1 Product Search

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roymustang11/InSAR-Benchmark-Lab/blob/main/notebooks/02_hyp3_or_opera_to_timeseries.ipynb)

This notebook starts the first real data path for the Central Valley subsidence benchmark: discovering OPERA Sentinel-1 surface displacement products without downloading large files into the repository.

The target collection is `OPERA_L3_DISP-S1_V1`. The notebook performs metadata-first discovery and writes a small local inventory table when live search is enabled.


## Scope

This notebook does three things:

1. loads the shared Central Valley study-area configuration,
2. defines an OPERA DISP-S1 metadata search using NASA Earthdata / CMR through `earthaccess`,
3. converts search results into a small inventory table suitable for later download, streaming, or product comparison.

It does **not** commit or download large NetCDF, HDF5, GeoTIFF, or Zarr products. Large products belong under ignored local directories such as `data/raw/` or `data/external/`.


## Source Notes

- OPERA DISP-S1 is the OPERA Level-3 surface displacement product derived from Sentinel-1. NASA Earthdata describes it as useful for deformation applications including subsidence, tectonics, and landslides.
- `earthaccess` searches NASA's Common Metadata Repository and can download or stream NASA Earthdata products after authentication.
- This notebook starts with metadata search only. Download/authentication will be added after the first target products are selected.

Useful references:

- [NASA Earthdata OPERA project](https://www.earthdata.nasa.gov/data/projects/opera)
- [OPERA products documentation](https://nasa-opera.github.io/docs/products/)
- [earthaccess search guide](https://earthaccess.readthedocs.io/en/latest/user_guide/search/)


## Setup

The notebook is safe to execute by default. `RUN_LIVE_SEARCH` is set to `False` so automated checks and first-time readers do not depend on network access. Set it to `True` when you want to query Earthdata live.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/roymustang11/InSAR-Benchmark-Lab.git"
RUN_LIVE_SEARCH = False

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    project_root = Path("/content/InSAR-Benchmark-Lab")
    if not project_root.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(project_root)])
else:
    cwd = Path.cwd().resolve()
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

config_path = project_root / "configs" / "central_valley_subsidence.yml"
inventory_dir = project_root / "data" / "interim" / "opera_disp_s1"
inventory_path = inventory_dir / "central_valley_search_inventory.csv"

print(f"Project root: {project_root}")
print(f"Live Earthdata search enabled: {RUN_LIVE_SEARCH}")


## Load Study-Area Configuration


In [ ]:
from insar_benchmark_lab.config import load_study_area_config

config = load_study_area_config(config_path)
bbox = config.region
search_bounding_box = (bbox["west"], bbox["south"], bbox["east"], bbox["north"])
search_temporal = (config.time_window["start"], config.time_window["end"])

print(config.name)
print("Bounding box:", search_bounding_box)
print("Temporal window:", search_temporal)


## Define OPERA DISP-S1 Search

The Earthdata query is deliberately small. It asks for a limited number of granules intersecting the Central Valley bounding box and configured time window.


In [ ]:
OPERA_DISP_S1_SHORT_NAME = "OPERA_L3_DISP-S1_V1"
MAX_GRANULES = 10

search_kwargs = {
    "short_name": OPERA_DISP_S1_SHORT_NAME,
    "bounding_box": search_bounding_box,
    "temporal": search_temporal,
    "count": MAX_GRANULES,
}

search_kwargs


## Run Metadata Search

Set `RUN_LIVE_SEARCH = True` in the setup cell to query NASA Earthdata. Metadata search should not require downloading large products. Download cells will be added only after the first target granules are selected.


In [ ]:
import pandas as pd

if RUN_LIVE_SEARCH:
    import earthaccess

    results = earthaccess.search_data(**search_kwargs)
else:
    results = []
    print("RUN_LIVE_SEARCH is False; skipping earthaccess.search_data.")

print(f"Granules returned: {len(results)}")


## Build Inventory Table

The inventory is intentionally small and text-based. It can be committed only if it contains metadata and no large product payloads. By default this notebook does not write anything because live search is disabled.


In [ ]:
def granule_to_record(granule):
    umm = granule.get("umm", {})
    collection = umm.get("CollectionReference", {})
    temporal = umm.get("TemporalExtent", {}).get("RangeDateTime", {})
    data_links = granule.data_links() if hasattr(granule, "data_links") else []
    return {
        "granule_id": umm.get("GranuleUR"),
        "short_name": collection.get("ShortName") or OPERA_DISP_S1_SHORT_NAME,
        "version": collection.get("Version"),
        "begin_time": temporal.get("BeginningDateTime"),
        "end_time": temporal.get("EndingDateTime"),
        "size_mb": granule.get("size"),
        "first_data_link": data_links[0] if data_links else None,
    }

inventory = pd.DataFrame([granule_to_record(granule) for granule in results])
inventory


In [ ]:
if RUN_LIVE_SEARCH and not inventory.empty:
    inventory_dir.mkdir(parents=True, exist_ok=True)
    inventory.to_csv(inventory_path, index=False)
    print(f"Wrote metadata inventory: {inventory_path}")
else:
    print("No inventory written. Enable RUN_LIVE_SEARCH and rerun after checking the search parameters.")


## Next Steps

After confirming the OPERA DISP-S1 search returns suitable Central Valley granules:

1. inspect the first NetCDF/Zarr product metadata without committing the product,
2. identify displacement, temporal coherence, acquisition-date, and spatial-reference variables,
3. extract one small station or pixel-level time series for a controlled validation example,
4. compare that time series with GNSS in `03_mintpy_timeseries_validation.ipynb`.

If OPERA coverage or authentication blocks the first experiment, use ASF HyP3 as the fallback data path and keep the same inventory-table pattern.
